# Evaluación Parcial 1 - Machine Learning

**Asignatura:** MLY0100 Machine Learning  
**Integrantes:** Por completar  
**Dataset:** DS1-18-Datos-Properati.csv  
**Fecha:** Por completar


# Fase 1 - Comprensión del Negocio

## Contexto

El dataset utilizado contiene publicaciones de propiedades ubicadas en Argentina. Incluye información relacionada con la ubicación de los inmuebles, cantidad de ambientes, dormitorios y baños, superficies, precio, tipo de propiedad y tipo de operación, entre otras variables.

En esta primera etapa del proyecto se busca comprender estos datos antes de construir modelos de Machine Learning. Para esto se realizará un análisis exploratorio que permita conocer cómo están distribuidas las propiedades y sus precios, detectar datos faltantes y valores atípicos, y preparar posteriormente la información para que pueda ser utilizada en tareas de regresión y clasificación.

## Objetivo

Analizar las principales características de las propiedades del dataset para comprender su comportamiento, especialmente en relación con el precio, la ubicación, las superficies y el tipo de propiedad. También se busca detectar problemas de calidad de los datos, como valores faltantes y valores atípicos, para dejar la información preparada para etapas posteriores de Machine Learning.

## Pregunta analítica

¿Qué características de las propiedades, como la ubicación, la superficie y el tipo de propiedad, se relacionan con las diferencias observadas en sus precios de venta?

## Supuestos

Para comenzar el análisis se consideran los siguientes supuestos de trabajo, los cuales se revisarán durante la comprensión de los datos:

- Los registros corresponden a publicaciones de propiedades en venta en Argentina.
- El precio informado representa el valor publicado de cada propiedad y será analizado junto con variables como ubicación, superficie y tipo de propiedad.
- Los valores faltantes no se interpretarán como cero, sino que se estudiarán antes de decidir su tratamiento.
- Los valores muy altos o muy bajos no se eliminarán automáticamente, ya que primero se debe verificar si corresponden a errores o a propiedades reales con características diferentes.
- Las conclusiones de esta etapa se limitarán al dataset entregado y no se tomarán como una representación completa de todo el mercado inmobiliario argentino.

## Variable objetivo para regresión

Para una futura tarea de regresión se propone utilizar **price** como variable objetivo.

Esta variable es adecuada porque contiene valores numéricos continuos que representan el precio publicado de cada propiedad. El objetivo de un modelo de regresión sería estimar ese valor a partir de otras características del inmueble, por ejemplo su ubicación, superficie, cantidad de ambientes, dormitorios, baños y tipo de propiedad.

En esta evaluación no se entrenará todavía el modelo. En esta etapa solamente se identifica y justifica la variable objetivo, mientras se analiza y prepara el dataset.

## Variable objetivo para clasificación

Para una futura tarea de clasificación se propone utilizar **property_type** como variable objetivo.

Esta variable es adecuada porque representa categorías discretas de propiedades. Un modelo de clasificación podría intentar identificar el tipo de propiedad a partir de características como la superficie, cantidad de ambientes, dormitorios, baños, ubicación y precio.

Al igual que en el caso de regresión, en esta evaluación no se entrenará todavía el modelo. En esta etapa solamente se identifica y justifica el target de clasificación.


# Fase 2 - Comprensión de los Datos

## Carga de librerías y datos

Antes de analizar el dataset se importan las librerías que se utilizarán durante el trabajo. Luego se carga el archivo CSV que está dentro del ZIP entregado por el profesor. Se mantiene el archivo original sin modificaciones en esta etapa, porque primero necesitamos conocer su estructura y revisar la calidad de los datos.


In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Buscar el ZIP en ubicaciones comunes del proyecto o Google Colab
candidatos = []
for carpeta in [Path('.'), Path('..'), Path('/content')]:
    candidatos.extend(carpeta.glob('Evaluación Parcial 1 - Dataset*.zip'))

if not candidatos:
    raise FileNotFoundError('No se encontró el ZIP del dataset.')

ruta_zip = candidatos[0]

with zipfile.ZipFile(ruta_zip, 'r') as archivo_zip:
    nombre_csv = next(
        nombre for nombre in archivo_zip.namelist()
        if nombre.endswith('DS1-18-Datos-Properati.csv')
        and not nombre.startswith('__MACOSX')
    )
    with archivo_zip.open(nombre_csv) as archivo_csv:
        df = pd.read_csv(archivo_csv)

print('Dimensiones del dataset:', df.shape)
df.head()


### Resultado de la carga

El dataset contiene **146.660 registros y 19 columnas**. La carga se realizó correctamente y las primeras filas muestran variables relacionadas con fechas, ubicación, ambientes, superficies, precio y tipo de propiedad. En este punto todavía no se realizan cambios sobre los datos; primero se revisará su estructura en detalle.


## Inspección de la estructura

En este bloque se revisan los nombres de las columnas, tipos de datos, cantidad de valores no nulos y un resumen general del dataset. Esta revisión permite detectar desde el inicio variables que necesitan correcciones de tipo antes de comenzar la limpieza y el análisis estadístico.


In [ ]:
# Revisar columnas y tipos de datos
print('Columnas del dataset:')
print(df.columns.tolist())

print('\nTipos de datos:')
print(df.dtypes)

print('\nInformación general:')
df.info()

# Resumen de variables numéricas y categóricas
columnas_numericas = df.select_dtypes(include='number').columns.tolist()
columnas_texto = df.select_dtypes(include='object').columns.tolist()

print('\nVariables numéricas:', columnas_numericas)
print('Variables almacenadas como texto/object:', columnas_texto)

df.describe(include='all').T


### Hallazgos de la estructura

El dataset tiene **19 columnas**. Actualmente se observan **8 variables de tipo `float64` y 11 de tipo `object`**.

Las variables `start_date`, `end_date` y `created_on` están almacenadas como texto, aunque representan fechas. Más adelante será necesario convertirlas a `datetime` para trabajar con un tipo de dato adecuado.

También se observa que algunas variables numéricas tienen menos registros no nulos que el total de 146.660 filas. Por ejemplo, `lat`, `lon`, `bathrooms`, `surface_total` y `surface_covered` presentan datos faltantes. El tratamiento de estos valores se realizará en su sección correspondiente, sin modificarlos todavía.

Variables como `l1`, `l2`, `l3`, `currency`, `property_type` y `operation_type` representan información categórica, mientras que `title` y `description` corresponden principalmente a texto descriptivo.


## Estadísticos descriptivos

Para comprender mejor las variables numéricas más importantes del dataset se calcularán medidas de tendencia central y dispersión. Se revisarán la media, mediana, moda, desviación estándar, varianza e IQR. Estas medidas ayudan a observar qué tan concentrados o dispersos están los datos y permiten detectar diferencias entre valores típicos y valores extremos.


In [ ]:
variables_analisis = [
    'rooms', 'bedrooms', 'bathrooms',
    'surface_total', 'surface_covered', 'price'
]

estadisticos = pd.DataFrame({
    'media': df[variables_analisis].mean(),
    'mediana': df[variables_analisis].median(),
    'moda': df[variables_analisis].mode().iloc[0],
    'desviacion_estandar': df[variables_analisis].std(),
    'varianza': df[variables_analisis].var(),
    'Q1': df[variables_analisis].quantile(0.25),
    'Q3': df[variables_analisis].quantile(0.75)
})

estadisticos['IQR'] = estadisticos['Q3'] - estadisticos['Q1']
estadisticos.round(2)


### Hallazgos de los estadísticos

Los resultados muestran diferencias importantes entre algunas variables. En `rooms`, la media es aproximadamente **3,08** y la mediana es **3**, por lo que los valores centrales son bastante parecidos. En `bedrooms`, la media es aproximadamente **1,98** y la mediana es **2**.

En cambio, `surface_total` presenta una media cercana a **216,87 m²** y una mediana de **78 m²**. Esta diferencia indica que existen propiedades con superficies muy grandes que elevan la media. Algo parecido ocurre con `surface_covered`, cuya media es aproximadamente **112,82 m²** y su mediana es **68 m²**.

La variable `price` también presenta una diferencia clara entre la media y la mediana. El precio promedio es aproximadamente **USD 241.221**, mientras que la mediana es **USD 166.000**. Además, su desviación estándar es cercana a **USD 318.519**, lo que indica una dispersión alta entre los precios publicados.

Estos resultados sugieren que variables como precio y superficie pueden contener valores extremos o distribuciones asimétricas. En los siguientes apartados se revisarán sus distribuciones y outliers antes de tomar decisiones de limpieza.


## Distribuciones y visualizaciones

Ahora se revisan gráficamente algunas variables importantes. Para las visualizaciones de `price` y `surface_total` se utiliza el percentil 99 solamente como límite visual, con el fin de que los valores extremos no compriman los gráficos. En esta etapa **no se eliminan registros del dataset**.


### Distribución del precio

Se utiliza un histograma para observar la forma de la distribución de `price`. También se muestran la media y la mediana para comparar ambas medidas de tendencia central.


In [ ]:
limite_price = df['price'].quantile(0.99)
price_grafico = df.loc[df['price'] <= limite_price, 'price'].dropna()

plt.figure(figsize=(9, 5))
sns.histplot(price_grafico, bins=40, kde=True, color='tab:blue')
plt.axvline(df['price'].mean(), linestyle='--', label='Media')
plt.axvline(df['price'].median(), linestyle='-', label='Mediana')
plt.title('Distribución del precio de las propiedades (hasta percentil 99)')
plt.xlabel('Precio (USD)')
plt.ylabel('Cantidad de propiedades')
plt.legend()
plt.show()


El histograma confirma que `price` presenta una distribución asimétrica hacia la derecha. La media es mayor que la mediana, lo que es consistente con la presencia de propiedades de precios altos que empujan el promedio hacia arriba.


### Boxplots de precio y superficie total

Los boxplots permiten observar la dispersión de los datos y la presencia de valores alejados del rango central. Nuevamente se limita la visualización al percentil 99, sin modificar el dataset original.


In [ ]:
limite_superficie = df['surface_total'].quantile(0.99)

plt.figure(figsize=(9, 4))
sns.boxplot(x=df.loc[df['price'] <= limite_price, 'price'])
plt.title('Boxplot del precio (hasta percentil 99)')
plt.xlabel('Precio (USD)')
plt.ylabel('Distribución')
plt.show()

plt.figure(figsize=(9, 4))
sns.boxplot(x=df.loc[df['surface_total'] <= limite_superficie, 'surface_total'])
plt.title('Boxplot de superficie total (hasta percentil 99)')
plt.xlabel('Superficie total (m²)')
plt.ylabel('Distribución')
plt.show()


Los boxplots muestran varios valores alejados de los rangos centrales tanto en precio como en superficie. Estos casos todavía no se eliminarán, porque primero deben analizarse formalmente como posibles outliers.


### Tipos de propiedad

Se utiliza un gráfico de barras para conocer cuáles son los tipos de propiedad con mayor cantidad de publicaciones.


In [ ]:
tipos_propiedad = df['property_type'].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=tipos_propiedad.values, y=tipos_propiedad.index, color='tab:blue')
plt.title('10 tipos de propiedad con más publicaciones')
plt.xlabel('Cantidad de publicaciones')
plt.ylabel('Tipo de propiedad')
plt.show()


`Departamento` es la categoría más frecuente con **107.326 publicaciones**, seguida por `Casa` con **21.521** y `PH` con **14.298**. Esto muestra que el dataset está concentrado principalmente en departamentos, aspecto que deberá considerarse si más adelante se utiliza `property_type` como target de clasificación.


### Relación entre superficie total y precio

Para observar la relación entre dos variables numéricas se realiza un scatter plot. Se utiliza una muestra para mejorar la legibilidad y se limita la visualización al percentil 99 de ambas variables.


In [ ]:
datos_scatter = df[
    (df['surface_total'] <= limite_superficie) &
    (df['price'] <= limite_price)
][['surface_total', 'price']].dropna()

muestra_scatter = datos_scatter.sample(
    n=min(5000, len(datos_scatter)),
    random_state=42
)

plt.figure(figsize=(9, 5))
sns.scatterplot(data=muestra_scatter, x='surface_total', y='price', alpha=0.35, color='tab:blue')
plt.title('Relación entre superficie total y precio')
plt.xlabel('Superficie total (m²)')
plt.ylabel('Precio (USD)')
plt.show()


El scatter plot muestra una tendencia positiva general: al aumentar la superficie total, el precio tiende a aumentar. Sin embargo, existe bastante dispersión, por lo que la superficie por sí sola no explica completamente las diferencias de precio observadas.


### Correlación entre variables numéricas

Finalmente se construye un heatmap de correlaciones para observar relaciones lineales entre las principales variables numéricas del análisis.


In [ ]:
matriz_correlacion = df[variables_analisis].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(matriz_correlacion, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlación de variables numéricas')
plt.xlabel('Variables')
plt.ylabel('Variables')
plt.show()


### Hallazgos de las visualizaciones

La correlación más alta entre las variables explicativas seleccionadas aparece entre `rooms` y `bedrooms`, con un valor cercano a **0,87**. Respecto de `price`, la relación lineal más alta dentro de este grupo se observa con `bathrooms`, cercana a **0,56**, seguida por `rooms` y `bedrooms`.

En cambio, las correlaciones directas de `surface_total` y `surface_covered` con `price` son bajas al calcularlas sobre los datos originales. Esto puede estar influido por valores extremos y por la heterogeneidad entre ubicaciones y tipos de propiedad, por lo que todavía no corresponde concluir que la superficie sea poco importante. Primero se deben revisar missing values y outliers.


# Fase 3 - Preparación de los Datos

## Missing values

Antes de modificar los datos se revisará cuántos valores faltantes existen y qué porcentaje representan. También se observará si la ausencia de datos parece estar relacionada con otras variables conocidas del dataset. Esta revisión es necesaria para decidir después qué tratamiento aplicar sin eliminar información de forma automática.


In [ ]:
resumen_nulos = pd.DataFrame({
    'cantidad_nulos': df.isna().sum(),
    'porcentaje_nulos': (df.isna().mean() * 100).round(2)
})

resumen_nulos = resumen_nulos[resumen_nulos['cantidad_nulos'] > 0]
resumen_nulos = resumen_nulos.sort_values('porcentaje_nulos', ascending=False)
resumen_nulos


Se encontraron valores faltantes solamente en cinco variables. `surface_covered` tiene **21.614 nulos (14,74%)**, `surface_total` **20.527 (14,00%)**, `lon` **9.959 (6,79%)**, `lat` **9.925 (6,77%)** y `bathrooms` **5.957 (4,06%)**.


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(
    x=resumen_nulos['porcentaje_nulos'],
    y=resumen_nulos.index,
    color='tab:blue'
)
plt.title('Porcentaje de valores faltantes por variable')
plt.xlabel('Porcentaje de valores faltantes (%)')
plt.ylabel('Variable')
plt.show()


### Análisis del mecanismo de ausencia

Para aproximarnos al mecanismo de los missing values se compara la ausencia de algunas variables con `property_type`. Si el porcentaje de nulos cambia claramente según una variable observada, no sería razonable asumir de inmediato que los datos faltan completamente al azar.


In [ ]:
for columna in ['surface_total', 'surface_covered', 'bathrooms']:
    porcentaje_por_tipo = (
        df.assign(faltante=df[columna].isna())
          .groupby('property_type')['faltante']
          .mean()
          .mul(100)
          .sort_values(ascending=False)
          .round(2)
    )
    print(f'\nPorcentaje de nulos en {columna} según tipo de propiedad:')
    print(porcentaje_por_tipo.head(10))

lat_lon_ambos = (df['lat'].isna() & df['lon'].isna()).sum()
print('\nRegistros donde faltan lat y lon al mismo tiempo:', lat_lon_ambos)


### Interpretación de los missing values

Los porcentajes de ausencia cambian bastante según el tipo de propiedad. Por ejemplo, `surface_covered` falta en casi todos los registros de `Cochera`, `Lote`, `Depósito` y `Local comercial`, mientras que en `PH` el porcentaje de ausencia es cercano al 1%. `bathrooms` también falta con mucha mayor frecuencia en categorías como `Cochera`, `Depósito` y `Lote` que en `Departamento` o `PH`.

Además, en **9.925 registros** faltan `lat` y `lon` al mismo tiempo. Esto muestra que varias ausencias aparecen relacionadas entre sí o con características observadas de la publicación.

Por esta evidencia, para este trabajo se considerará que los missing values son **principalmente compatibles con un mecanismo MAR (Missing At Random)**, porque la probabilidad de ausencia parece depender de información observable como `property_type`. Esto es una interpretación de trabajo, no una demostración definitiva del mecanismo estadístico. El tratamiento concreto de cada variable se decidirá en el siguiente paso.


### Comparación con KNNImputer

Como alternativa a la imputación por mediana se prueba `KNNImputer` sobre una muestra de 3.000 registros y las variables `bathrooms`, `surface_total` y `surface_covered`. Se utiliza una muestra para evitar un costo computacional innecesariamente alto sobre las 146.660 filas.

La comparación se realiza solo para evaluar el comportamiento del método. La estrategia final seguirá siendo la mediana agrupada, porque es más simple de interpretar y mantiene una lógica relacionada con `property_type`.


In [ ]:
variables_knn = ['bathrooms', 'surface_total', 'surface_covered']
muestra_knn = df[variables_knn].sample(n=3000, random_state=42).copy()

knn_imputer = KNNImputer(n_neighbors=5)
muestra_knn_imputada = pd.DataFrame(
    knn_imputer.fit_transform(muestra_knn),
    columns=variables_knn,
    index=muestra_knn.index
)

comparacion_knn = pd.DataFrame({
    'media_original': muestra_knn.mean(),
    'media_knn': muestra_knn_imputada.mean(),
    'std_original': muestra_knn.std(),
    'std_knn': muestra_knn_imputada.std(),
})

comparacion_knn.round(2)


La prueba con `KNNImputer` permite completar los nulos manteniendo valores coherentes con observaciones cercanas. Sin embargo, para el dataset completo se prefiere la imputación por mediana agrupada porque es más transparente, reproducible y fácil de justificar según el tipo de propiedad y la zona.


### Tratamiento de los valores faltantes

Como la ausencia de datos parece estar relacionada con variables observadas, no se eliminarán filas de forma automática. Para `surface_total`, `surface_covered` y `bathrooms` se imputará la **mediana según `property_type`**. Se utiliza la mediana porque estas variables presentan valores extremos y esta medida es menos sensible a ellos que la media.

Para `lat` y `lon` se utilizará la **mediana según `l3`**, ya que `l3` representa una zona más específica y permite mantener una referencia geográfica aproximada sin reemplazar todos los datos por una única coordenada global.


In [ ]:
# Trabajamos sobre una copia para conservar el dataset original
df_limpio = df.copy()

# Imputación por mediana según tipo de propiedad
for columna in ['surface_total', 'surface_covered', 'bathrooms']:
    mediana_por_tipo = df_limpio.groupby('property_type')[columna].transform('median')
    df_limpio[columna] = df_limpio[columna].fillna(mediana_por_tipo)

# Imputación de coordenadas por mediana según zona l3
for columna in ['lat', 'lon']:
    mediana_por_zona = df_limpio.groupby('l3')[columna].transform('median')
    df_limpio[columna] = df_limpio[columna].fillna(mediana_por_zona)

print('Nulos restantes por columna:')
print(df_limpio.isna().sum()[df_limpio.isna().sum() > 0])
print('\nDimensiones después de la imputación:', df_limpio.shape)


### Resultado del tratamiento

Después de aplicar la imputación quedan **0 valores nulos** en el dataset y se conservan las **146.660 filas originales**. De esta forma no se pierde información por eliminación de registros y se mantiene el tamaño del conjunto de datos.

La decisión se tomó considerando la relación observada entre los missing values y variables como `property_type` y `l3`, además de la presencia de valores extremos en superficies y baños.


## Outliers

Ahora se analizarán los valores atípicos de las principales variables numéricas utilizando dos métodos: **IQR** y **Z-score**. El método IQR considera como posibles outliers los valores menores a `Q1 - 1,5 × IQR` o mayores a `Q3 + 1,5 × IQR`. El Z-score permite detectar valores alejados más de 3 desviaciones estándar de la media.

En esta etapa solo se detectan y comparan los posibles outliers. **Todavía no se eliminan ni reemplazan registros**, porque primero se debe evaluar su impacto.


In [ ]:
resumen_outliers = []

for columna in variables_analisis:
    q1 = df_limpio[columna].quantile(0.25)
    q3 = df_limpio[columna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    mascara_iqr = (
        (df_limpio[columna] < limite_inferior) |
        (df_limpio[columna] > limite_superior)
    )

    z_scores = np.abs(stats.zscore(df_limpio[columna]))
    cantidad_zscore = int((z_scores > 3).sum())

    resumen_outliers.append({
        'variable': columna,
        'limite_inferior_iqr': limite_inferior,
        'limite_superior_iqr': limite_superior,
        'outliers_iqr': int(mascara_iqr.sum()),
        'porcentaje_iqr': round(mascara_iqr.mean() * 100, 2),
        'outliers_zscore': cantidad_zscore,
        'porcentaje_zscore': round(cantidad_zscore / len(df_limpio) * 100, 2)
    })

resumen_outliers = pd.DataFrame(resumen_outliers)
resumen_outliers


### Visualización de los posibles outliers

Se vuelven a utilizar boxplots sobre el dataset ya imputado para observar visualmente la dispersión de `price` y `surface_total`.


In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(x=df_limpio['price'])
plt.title('Boxplot de price después de tratar valores faltantes')
plt.xlabel('Precio (USD)')
plt.ylabel('Distribución')
plt.show()

plt.figure(figsize=(9, 4))
sns.boxplot(x=df_limpio['surface_total'])
plt.title('Boxplot de surface_total después de tratar valores faltantes')
plt.xlabel('Superficie total (m²)')
plt.ylabel('Distribución')
plt.show()


### Hallazgos de outliers

El método IQR detecta **10.982 posibles outliers en `price` (7,49%)** y **19.290 en `surface_total` (13,15%)**. En `surface_covered` identifica **9.552 casos (6,51%)**.

El Z-score es más restrictivo en estas variables y detecta **2.335 casos en `price` (1,59%)**, **350 en `surface_total` (0,24%)** y **205 en `surface_covered` (0,14%)**.

La diferencia entre ambos métodos confirma que las variables de precio y superficie tienen distribuciones fuertemente asimétricas. Por eso, marcar un registro como outlier estadístico no significa automáticamente que sea un error. En el contexto inmobiliario pueden existir propiedades reales con precios o superficies muy superiores al resto.


### Tratamiento de outliers

El método IQR marca una cantidad importante de registros, especialmente en `price` y las superficies. Como en el mercado inmobiliario pueden existir propiedades realmente más grandes o caras que el resto, eliminar todos esos registros podría hacer que el dataset pierda información válida.

Por esta razón se tratarán solamente los valores **extremos de la cola superior**, utilizando el **percentil 99** como límite. Los valores que superen ese percentil se reemplazarán por el valor del percentil 99 de su variable. De esta forma se reduce el efecto de los casos más extremos sin eliminar filas completas.


In [ ]:
variables_outliers = [
    'rooms', 'bedrooms', 'bathrooms',
    'surface_total', 'surface_covered', 'price'
]

resumen_tratamiento = []

for columna in variables_outliers:
    limite_p99 = df_limpio[columna].quantile(0.99)
    maximo_antes = df_limpio[columna].max()
    cantidad_ajustada = int((df_limpio[columna] > limite_p99).sum())

    df_limpio[columna] = df_limpio[columna].clip(upper=limite_p99)

    resumen_tratamiento.append({
        'variable': columna,
        'limite_percentil_99': limite_p99,
        'valores_ajustados': cantidad_ajustada,
        'maximo_antes': maximo_antes,
        'maximo_despues': df_limpio[columna].max()
    })

resumen_tratamiento = pd.DataFrame(resumen_tratamiento)
resumen_tratamiento.round(2)


### Resultado del tratamiento de outliers

El tratamiento afecta aproximadamente al 1% superior de cada variable. Los límites utilizados son: `rooms` **7**, `bedrooms` **5**, `bathrooms` **5**, `surface_total` **1.441,64 m²**, `surface_covered` **405 m²** y `price` **USD 1.450.000**.

Por ejemplo, el valor máximo original de `price` era superior a **USD 32 millones** y el máximo de `surface_total` superaba los **193 mil m²**. Después del ajuste, esos valores quedan limitados al percentil 99. Se mantienen las **146.660 filas**, por lo que no se pierden registros completos.

No se eliminan todos los valores señalados por IQR, porque varios pueden corresponder a propiedades reales. El objetivo de este tratamiento es controlar solamente los casos más extremos para reducir su influencia en los análisis posteriores.


## Normalización y estandarización

Antes de aplicar una técnica se revisa la asimetría de las variables numéricas después del tratamiento de outliers. Las variables `rooms` y `bedrooms` presentan una asimetría menor que el resto, mientras que `bathrooms`, `surface_total`, `surface_covered` y `price` siguen mostrando una asimetría positiva más marcada.

Por este motivo se aplicará **StandardScaler** a `rooms` y `bedrooms`, y **MinMaxScaler** a `bathrooms`, `surface_total`, `surface_covered` y `price`. Las columnas originales se conservarán para mantener la interpretación en sus unidades originales, y las transformaciones se guardarán en nuevas columnas.


In [ ]:
asimetria = df_limpio[variables_analisis].skew().round(3)
print('Asimetría después del tratamiento de outliers:')
print(asimetria)


Los valores de asimetría obtenidos son aproximadamente: `rooms` **0,73**, `bedrooms` **0,53**, `bathrooms` **1,63**, `surface_total` **4,13**, `surface_covered` **2,27** y `price` **3,23**. Esto respalda el uso de una técnica distinta según el comportamiento observado de cada grupo de variables.


In [ ]:
df_escalado = df_limpio.copy()

variables_standard = ['rooms', 'bedrooms']
variables_minmax = ['bathrooms', 'surface_total', 'surface_covered', 'price']

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

columnas_standard = [f'{col}_std' for col in variables_standard]
columnas_minmax = [f'{col}_minmax' for col in variables_minmax]

df_escalado[columnas_standard] = standard_scaler.fit_transform(
    df_limpio[variables_standard]
)

df_escalado[columnas_minmax] = minmax_scaler.fit_transform(
    df_limpio[variables_minmax]
)

df_escalado[columnas_standard + columnas_minmax].head()


### Comparación antes y después

Para comprobar el efecto de las transformaciones se comparan media, desviación estándar, mínimo y máximo antes y después del escalamiento.


In [ ]:
antes = df_limpio[variables_analisis].agg(['mean', 'std', 'min', 'max']).T

despues_standard = df_escalado[columnas_standard].agg(
    ['mean', 'std', 'min', 'max']
).T

despues_minmax = df_escalado[columnas_minmax].agg(
    ['mean', 'std', 'min', 'max']
).T

print('Estadísticos antes del escalamiento:')
display(antes.round(3))

print('Variables estandarizadas:')
display(despues_standard.round(3))

print('Variables normalizadas:')
display(despues_minmax.round(3))


### Resultado del escalamiento

Después de aplicar `StandardScaler`, las columnas `rooms_std` y `bedrooms_std` quedan centradas aproximadamente en **0** y con desviación estándar cercana a **1**.

Las columnas generadas con `MinMaxScaler` quedan dentro del rango **0 a 1**. Esta transformación cambia la escala de los datos, pero no elimina por sí sola la asimetría de la distribución.

Se mantienen también las variables originales para conservar valores interpretables, como precio en USD y superficies en m². Las nuevas columnas escaladas quedan disponibles para etapas posteriores de modelado.


## Dataset final

Antes de cerrar la preparación de los datos se corrigen los tipos de algunas variables. Las columnas de fecha se convierten a tipo `datetime64[s]`. Se utiliza precisión de segundos porque `end_date` contiene el valor `9999-12-31` en algunos registros, valor que puede representar una fecha de término abierta y que no cabe en el rango habitual de `datetime64[ns]` de pandas.

Además, las variables que representan categorías se convierten al tipo `category`. Esto permite que el dataset refleje mejor la naturaleza de cada variable sin cambiar su contenido.


In [ ]:
df_final = df_escalado.copy()

# Conversión de variables de fecha
columnas_fecha = ['start_date', 'end_date', 'created_on']
for columna in columnas_fecha:
    df_final[columna] = df_final[columna].astype('datetime64[s]')

# Conversión de variables categóricas
columnas_categoricas = [
    'l1', 'l2', 'l3', 'currency',
    'property_type', 'operation_type'
]

for columna in columnas_categoricas:
    df_final[columna] = df_final[columna].astype('category')

print('Dimensiones del dataset final:', df_final.shape)
print('Valores nulos totales:', int(df_final.isna().sum().sum()))
print('\nTipos de datos finales:')
print(df_final.dtypes)


### Verificación del dataset final

El dataset final mantiene las **146.660 filas** y queda con **25 columnas**, ya que a las 19 variables originales se agregaron 6 columnas escaladas.

Después de la imputación, el tratamiento de outliers, el escalamiento y la corrección de tipos, el dataset queda con **0 valores nulos**. Las fechas quedan almacenadas como variables temporales y las principales variables categóricas quedan identificadas con el tipo `category`.

En este punto el dataset ya está preparado para su exportación y para etapas posteriores de Machine Learning.


### Exportación del dataset preparado

Como último paso de la preparación se exporta el dataset final a un archivo CSV. Se utiliza `index=False` para evitar agregar una columna adicional con el índice de pandas.


In [ ]:
ruta_salida = Path('DS1-18-Datos-Properati-preparado.csv')

df_final.to_csv(ruta_salida, index=False)

print('Dataset exportado en:', ruta_salida.resolve())
print('Archivo creado correctamente:', ruta_salida.exists())


El archivo `DS1-18-Datos-Properati-preparado.csv` contiene el resultado de la Fase 3 y puede utilizarse posteriormente como base para las etapas de modelado.


## Conclusiones

El análisis permitió comprender la estructura y el comportamiento general del dataset de propiedades antes de utilizarlo en modelos de Machine Learning. Se trabajó con **146.660 registros**, revisando variables relacionadas con precio, superficies, cantidad de ambientes, dormitorios, baños, ubicación y tipo de propiedad.

Respecto de la pregunta analítica, los resultados muestran que el precio de una propiedad no se relaciona con una sola característica. La relación entre `surface_total` y `price` presenta una tendencia positiva, pero con una dispersión importante. Además, entre las variables numéricas revisadas, `bathrooms` presenta una de las relaciones lineales más altas con `price`, seguida por variables como `rooms` y `bedrooms`. Esto indica que las características físicas de la propiedad deben analizarse de manera conjunta.

También se observó que `price`, `surface_total` y `surface_covered` presentan distribuciones asimétricas y valores extremos. Por este motivo no se eliminaron automáticamente todos los registros marcados como outliers. Se limitaron solamente los valores de la cola superior mediante el percentil 99, conservando todas las filas del dataset.

En cuanto a los valores faltantes, se detectaron nulos en `surface_covered`, `surface_total`, `lat`, `lon` y `bathrooms`. Su comportamiento mostró diferencias según variables observadas como `property_type`, por lo que se trabajó con la hipótesis de que son principalmente compatibles con un mecanismo MAR. Los valores fueron imputados utilizando medianas agrupadas y el dataset quedó finalmente con **0 valores nulos**.

El tipo de propiedad también es relevante para futuras tareas de clasificación. `Departamento` concentra la mayoría de las publicaciones, seguido por `Casa` y `PH`, por lo que existe un desbalance entre categorías que deberá considerarse si posteriormente se entrena un modelo utilizando `property_type` como variable objetivo.

Finalmente, se aplicaron técnicas de normalización y estandarización según el comportamiento de las variables, manteniendo también las columnas originales para conservar su interpretación. El dataset final queda con **146.660 filas y 25 columnas**, con tipos de datos corregidos y variables escaladas disponibles para etapas posteriores.

Con esto se completan las tres primeras fases trabajadas de CRISP-DM: **Comprensión del Negocio, Comprensión de los Datos y Preparación de los Datos**. El conjunto queda preparado para continuar posteriormente con una tarea de regresión utilizando `price` y una tarea de clasificación utilizando `property_type`.
